In [6]:
# Imports
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

In [7]:
# Load data
test = pd.read_csv('/kaggle/input/datasets/deveshtiwari123/faulty-classification-binary/TEST.csv')
train = pd.read_csv('/kaggle/input/datasets/deveshtiwari123/faulty-classification-binary/TRAIN.csv')

# Clean headers
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

In [8]:
# Check shapes
train.shape, test.shape

((43776, 48), (10944, 48))

In [9]:
# Check data types and nulls
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43776 entries, 0 to 43775
Data columns (total 48 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   F01     43776 non-null  float64
 1   F02     43776 non-null  float64
 2   F03     43776 non-null  float64
 3   F04     43776 non-null  float64
 4   F05     43776 non-null  float64
 5   F06     43776 non-null  float64
 6   F07     43776 non-null  float64
 7   F08     43776 non-null  float64
 8   F09     43776 non-null  float64
 9   F10     43776 non-null  float64
 10  F11     43776 non-null  float64
 11  F12     43776 non-null  float64
 12  F13     43776 non-null  float64
 13  F14     43776 non-null  float64
 14  F15     43776 non-null  float64
 15  F16     43776 non-null  float64
 16  F17     43776 non-null  float64
 17  F18     43776 non-null  float64
 18  F19     43776 non-null  float64
 19  F20     43776 non-null  float64
 20  F21     43776 non-null  float64
 21  F22     43776 non-null  float64
 22

In [10]:
# View top rows
train.head()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
0,0.185570,0.004568,0.005362,0.003335,0.005415,0.004895,0.012764,0.120138,0.140450,3.361753,...,0.041526,-0.230857,0.003310,0.042250,0.005975,0.002104,0.013878,0.001518,0.011347,0
1,0.369536,0.003983,0.003386,0.004902,0.007570,0.012136,0.118050,0.323925,0.132093,2.766117,...,-0.141285,-6.222857,0.834177,0.227968,0.018463,-0.020487,0.001246,0.007489,0.008724,1
2,0.602510,0.008442,0.012961,0.012870,0.046885,0.115401,0.065688,0.306677,0.498805,4.521201,...,0.011334,10.335251,-0.276614,-0.198900,-0.012756,0.014286,-0.001866,0.002687,0.013452,1
3,0.347957,0.064721,0.013611,0.011541,0.006492,0.008690,0.013192,0.164553,0.298665,3.170847,...,0.190479,2.864912,-1.921939,0.891690,1.108098,0.635431,0.081368,-0.000225,0.009166,0
4,0.233653,0.012217,0.010088,0.022095,0.026040,0.015062,0.016063,0.084648,0.213367,8.150943,...,0.203164,0.001812,-0.092731,0.005280,-0.213985,0.032195,0.002081,0.028930,-0.025912,1


In [11]:
# View top rows
train.describe()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
count,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,...,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000
mean,0.567525,0.014285,0.015551,0.026779,0.087637,0.155142,0.150656,0.255644,0.361146,3.762811,...,0.023477,-0.039795,0.005846,-0.000089,0.003215,0.001007,0.004599,0.003981,-0.006780,0.395445
std,0.738884,0.019607,0.022504,0.050725,0.188394,0.377655,0.339948,0.354816,0.440134,1.271343,...,0.707224,8.462749,1.631328,1.103486,0.354742,0.171935,0.201611,0.202130,0.562031,0.488952
min,0.100025,0.003001,0.000752,0.000682,0.000835,0.000854,0.000702,0.001399,0.001164,0.038349,...,-22.647806,-28.026880,-9.109769,-9.275399,-10.607320,-5.910117,-11.029342,-18.853842,-44.878769,0.000000
25%,0.198843,0.004778,0.005437,0.006290,0.012050,0.014112,0.015084,0.072098,0.128780,3.192588,...,-0.036589,-0.254039,-0.117789,-0.015759,-0.011839,-0.007214,-0.005226,-0.000826,-0.008882,0.000000
50%,0.288778,0.007988,0.008905,0.012619,0.023681,0.032396,0.029157,0.119095,0.214665,3.604579,...,-0.000210,-0.000757,-0.002891,-0.000111,-0.000045,-0.000170,0.000295,0.000092,-0.000083,0.000000
75%,0.528760,0.014382,0.016610,0.027599,0.063469,0.115950,0.116992,0.265227,0.381230,4.035824,...,0.043276,0.171122,0.107351,0.013399,0.009806,0.005892,0.006058,0.002405,0.007418,1.000000
max,12.779628,0.199925,0.506419,0.851009,5.017180,7.249545,6.556998,5.960009,5.085546,50.624777,...,89.378571,28.471415,10.907778,9.374269,11.399358,9.139964,18.623611,8.960172,40.818882,1.000000


In [12]:
# Check duplicate rows
train.duplicated().sum()

np.int64(738)

In [13]:
# Drop duplicates
train = train.drop_duplicates().reset_index(drop=True)

In [14]:
# Define base features
base_features = [col for col in train.columns if col.startswith('F')]
y = train['Class']
test_ids = test['ID']

In [15]:
# Row-wise stats
def generate_row_features(df, features):
    df_fe = df.copy()
    df_fe['row_mean'] = df[features].mean(axis=1)
    df_fe['row_std'] = df[features].std(axis=1)
    df_fe['row_min'] = df[features].min(axis=1)
    df_fe['row_max'] = df[features].max(axis=1)
    df_fe['row_skew'] = df[features].skew(axis=1)
    df_fe['row_kurt'] = df[features].kurtosis(axis=1)
    df_fe['zero_count'] = (df[features] == 0).sum(axis=1)
    return df_fe

train_fe = generate_row_features(train, base_features)
test_fe = generate_row_features(test, base_features)

In [16]:
# PCA & KMeans features
all_data = pd.concat([train[base_features], test[base_features]], axis=0)

pca = PCA(n_components=5, random_state=42)
pca.fit(all_data)
for i in range(5):
    train_fe[f'pca_{i}'] = pca.transform(train[base_features])[:, i]
    test_fe[f'pca_{i}'] = pca.transform(test[base_features])[:, i]

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(all_data)
train_fe['cluster_dist'] = kmeans.transform(train[base_features]).min(axis=1)
test_fe['cluster_dist'] = kmeans.transform(test[base_features]).min(axis=1)

final_features = [col for col in train_fe.columns if col not in ['ID', 'Class']]
X = train_fe[final_features]
X_test = test_fe[final_features]

In [18]:
# CV Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof = np.zeros(len(train))
lgb_oof = np.zeros(len(train))
cat_oof = np.zeros(len(train))

xgb_test_preds = np.zeros(len(test))
lgb_test_preds = np.zeros(len(test))
cat_test_preds = np.zeros(len(test))

In [19]:
# Training loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=2000, learning_rate=0.03, max_depth=6, 
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
        early_stopping_rounds=100, random_state=42, n_jobs=-1
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    # LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=31, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1
    )
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    
    # CatBoost
    cat_model = CatBoostClassifier(
        iterations=2000, learning_rate=0.03, depth=6,
        eval_metric='AUC', random_seed=42, thread_count=-1, verbose=0
    )
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    
    # Predict validation
    xgb_oof[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    lgb_oof[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    cat_oof[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    
    # Predict test
    xgb_test_preds += xgb_model.predict_proba(X_test)[:, 1] / skf.n_splits
    lgb_test_preds += lgb_model.predict_proba(X_test)[:, 1] / skf.n_splits
    cat_test_preds += cat_model.predict_proba(X_test)[:, 1] / skf.n_splits

In [20]:
# Ensemble & optimize threshold
ensemble_oof = (xgb_oof + lgb_oof + cat_oof) / 3
ensemble_test_preds = (xgb_test_preds + lgb_test_preds + cat_test_preds) / 3

best_thresh = 0.50
best_acc = 0
thresholds = np.linspace(0.01, 0.99, 99)

for thresh in thresholds:
    acc = accuracy_score(y, (ensemble_oof > thresh).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_thresh = thresh

# Display optimized accuracy
best_acc, best_thresh

(0.9881964775314838, np.float64(0.4))

In [23]:
# Generate submission
submission = pd.DataFrame({
    'ID': test_ids,
    'CLASS': (ensemble_test_preds > best_thresh).astype(int)
})

submission.to_csv('OUTPUT.csv', index=False)
submission.head()

,ID,CLASS
0,1,1
1,2,0
2,3,1
3,4,0
4,5,0
